# tool_02 单吸附位点构建（参考 tool + workflow）

## 目标
- 参考 `DFT/tool/H2O_单吸附模型构建.ipynb` 的位点构建逻辑
- 参考 `DFT/workflow/GO_1body_builder_ase.py` 的 CO/CHO/COOH 分子来源
- 分子结构优先使用 ASE 标准来源
- 构建时必须指定 `layer`（从顶层开始计数）
- 仅生成 `.vasp` 结构文件，不做提交调度


In [ ]:
from pathlib import Path
import sys
import tarfile
from fnmatch import fnmatch
from ase.io import read, write


In [ ]:
# 从 build/shared 加载配置
_p = Path.cwd()
while _p != _p.parent:
    if (_p / 'shared' / 'config.py').exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent

from shared.load_config import load_config
from shared.single_adsorption_builder import create_single_adsorption_structure, SUPPORTED_SITES

cfg = load_config()
go_root = Path(cfg.GO_ROOT)
surface_poscar = Path(cfg.SURFACE_POSCAR)
print(f'GO_ROOT: {go_root}')
print(f'SURFACE_POSCAR: {surface_poscar}')

In [ ]:
# ---------- 用户参数（先改这里）----------
# 吸附物完全由任务清单决定：按本次任务填写 species_list
species_list = ['H', 'O', 'OH', 'H2', 'H2O', 'CO', 'CHO', 'COOH']
adsorption_distance = 2.0  # workflow 默认
sites = ['hollow', 'top', 'bridge']
layer_from_top = 1         # 必填：1=顶层，2=次顶层，...
center_layer_from_top = 1  # 位点中心参考层：默认第一层（顶层）

# 母 folder 设置：默认使用 GO_ROOT；特殊情况可改名
parent_folder_name = 'GO_ROOT'  # 默认；例如改成 'GO_custom' 则输出到 GO_ROOT 同级目录/GO_custom

export_test_copy = False    # True 时复制一份到 TEST 供人工检查
export_atom_index = True    # True 时为每个结构导出原子标号清单

# 打包设置：默认打包母 folder 下全部 .vasp
package_vasp = True
package_all_vasp = True
package_patterns = ['*.vasp']  # package_all_vasp=False 时生效，例如 ['*H2O*.vasp', '*CO*.vasp']
package_name = None            # None => <parent_folder_name>_vasp.tar.gz

assert set(sites).issubset(set(SUPPORTED_SITES)), f'不支持位点: {set(sites) - set(SUPPORTED_SITES)}'
assert isinstance(layer_from_top, int) and layer_from_top >= 1, 'layer_from_top 必须是 >=1 的整数'
assert isinstance(center_layer_from_top, int) and center_layer_from_top >= 1, 'center_layer_from_top 必须是 >=1 的整数'

In [ ]:
surface_atoms = read(str(surface_poscar), format='vasp')
created = []
index_files = []

# 母 folder 解析：默认 GO_ROOT；否则用 GO_ROOT 同级自定义目录
if parent_folder_name == 'GO_ROOT':
    parent_dir = go_root
else:
    parent_dir = go_root.parent / parent_folder_name
parent_dir.mkdir(parents=True, exist_ok=True)

print(f'使用 layer_from_top = {layer_from_top}')
print(f'位点中心参考层 center_layer_from_top = {center_layer_from_top}（默认第一层中心）')
print(f'结构母 folder: {parent_dir}')

for species in species_list:
    for site in sites:
        atoms = create_single_adsorption_structure(
            surface_atoms=surface_atoms,
            species=species,
            site=site,
            layer_from_top=layer_from_top,
            center_layer_from_top=center_layer_from_top,
            adsorption_distance=adsorption_distance,
        )

        # 结构直接输出到母 folder 根目录
        fname = f'job_POSCAR_{species}_{site}.vasp'
        fpath = parent_dir / fname
        write(str(fpath), atoms, format='vasp', vasp5=True)
        created.append(fpath)

        # 为后续任务准备：导出每个原子的标号、元素和坐标
        if export_atom_index:
            idx_path = parent_dir / f'job_POSCAR_{species}_{site}_atom_index.txt'
            symbols = atoms.get_chemical_symbols()
            positions = atoms.get_positions()
            with open(idx_path, 'w', encoding='utf-8') as f:
                f.write('# idx symbol x y z\n')
                for i, (sym, pos) in enumerate(zip(symbols, positions), start=1):
                    f.write(f'{i:4d} {sym:>3s} {pos[0]:12.6f} {pos[1]:12.6f} {pos[2]:12.6f}\n')
            index_files.append(idx_path)

        if export_test_copy:
            test_dir = go_root / 'test_absorption' / f'absorption_single_{species}'
            test_dir.mkdir(parents=True, exist_ok=True)
            write(str(test_dir / fname), atoms, format='vasp', vasp5=True)

print(f'✅ 已生成 {len(created)} 个 .vasp 结构')
for p in created[:10]:
    print(' -', p)
if len(created) > 10:
    print(f' ... 其余 {len(created)-10} 个已写入')

if export_atom_index:
    print(f'✅ 已生成 {len(index_files)} 份原子标号清单 (*.txt)')

# 打包：默认打包母 folder 下全部 .vasp
if package_vasp:
    vasp_files = sorted(parent_dir.glob('*.vasp'))
    if not package_all_vasp:
        selected = []
        for vf in vasp_files:
            if any(fnmatch(vf.name, pat) for pat in package_patterns):
                selected.append(vf)
        vasp_files = selected

    if not vasp_files:
        print('⚠ 未匹配到可打包的 .vasp 文件')
    else:
        tar_name = package_name or f"{parent_dir.name}_vasp.tar.gz"
        tar_path = parent_dir / tar_name
        with tarfile.open(tar_path, 'w:gz') as tar:
            for vf in vasp_files:
                tar.add(vf, arcname=vf.name)
        print(f'📦 已打包 {len(vasp_files)} 个 .vasp -> {tar_path}')

## 输出说明
- 吸附物种由**任务清单**决定：每次任务只填写并生成 `species_list` 中指定的物种
- 分子结构优先使用 ASE 标准来源（H2/H2O/OH/CO/CHO/COOH 等）
- 构建时必须指定 `layer_from_top`（1=顶层，2=次顶层，...）
- 高对称位点搜索中心默认优先使用**第一层 layer 中心**：`center_layer_from_top=1`
- `.vasp` 结构文件**直接输出到母 folder 根目录**（默认母 folder 为 `GO_ROOT`）
- 母 folder 可灵活改名：`parent_folder_name='GO_ROOT'`（默认）或自定义名字
- 每个结构可同步输出原子标号清单：`job_POSCAR_<species>_<site>_atom_index.txt`（序号、元素、坐标）
- 打包默认行为：打包母 folder 下全部 `.vasp`；可通过 `package_all_vasp=False` + `package_patterns` 只打包部分
- 本 tool 仅生成 build 所需的 `.vasp` 结构与标号清单，不生成 POTCAR/KPOINTS/INCAR
- 后续由 Global_optimization 或对应输入文件生成工具继续处理
